# Sweep: vary a design parameter and a seed

This notebook runs the FIFO of the examples through `yosys_fpga+nextpnr` for an ECP5 device. It
changes one design parameter (`LOG2_DEPTH`) and one setting of `nextpnr` (the seed). It collects the
results in a table and plots LUTs against `fmax`. At the end, it reads the same results again from
the run root.

The notebook needs Yosys, `nextpnr-ecp5`, pandas and Plotly.

In [ ]:
import os
from pathlib import Path

import xeda

EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(f"No examples in {EXAMPLES}. Set XEDA_EXAMPLES to the examples directory.")

WORK = Path("notebook-output").resolve()
ws = xeda.Workspace(WORK / "xeda_run", hashed_run_dirs=True)

## Build the specs

A sweep takes any iterable of `Spec`. You build the specs with plain Python.

`Design.with_updates` returns a new design and leaves the original as it is. It accepts a dotted
key. The last point has a clock period of 2 ns. This clock is too fast for the device, so that
point does not pass.

In [ ]:
from collections import Counter
from itertools import product

import pandas as pd
import plotly.graph_objects as go

from xeda import Design, Spec
from xeda.errors import RunReplacedError, SpecError
from xeda.introspect import results_info

base = Design.load(EXAMPLES / "sv/fifo/fifo.xeda.yaml")


def point(depth: int, seed: int, period: str = "10 ns") -> Spec:
    return Spec(
        "yosys_fpga+nextpnr",
        base.with_updates({"rtl.parameters.LOG2_DEPTH": depth}),
        settings={
            "yosys_fpga": {"fpga.part": "LFE5U-85F-6BG381C", "clock.period": period},
            "nextpnr": {"lpf_allow_unconstrained": True, "seed": seed},
        },
    )


specs = [point(depth, seed) for depth, seed in product((3, 4, 5), (1, 2))]
specs.append(point(4, 1, period="2 ns"))
print(len(specs), "points")

## Use hashed run directories

These specs differ in a design parameter. With the default layout, they would all use the run
directory `fifo/yosys_fpga`. Xeda refuses this before it runs anything.

In [ ]:
plain = xeda.Workspace(WORK / "plain")
try:
    plain.sweep(specs)
except SpecError as error:
    print(error.code)
    print(error.points)

A workspace with `hashed_run_dirs=True` gives each configuration its own directory. The first cell
of this notebook made `ws` that way.

## Run the sweep

Xeda plans all points before it runs one. `max_workers=2` runs two nodes at a time, in two worker
processes.

In [ ]:
sweep = ws.sweep(specs, max_workers=2)
print({outcome.value: count for outcome, count in sweep.counts().items()})

Points that need the same node run it once. The seed does not change the synthesis, so the two
seeds of one depth share one `yosys_fpga` run. The first point of the group reports `ran`. The
others report `shared`. The point with the 2 ns clock has its own synthesis, because the clock is
a setting of `yosys_fpga`.

In [ ]:
dispositions = Counter(p.result.nodes["yosys_fpga"].disposition.value for p in sweep)
print(dict(sorted(dispositions.items())))

## Make a table

`to_pandas` gives one row for each point, in input order. You choose the metrics. A metric is
named `<node>.<metric>`, for example `nextpnr.fmax`. The sweep keeps the points that did not pass.
A point has `None` for a metric that the record of its node does not hold.

In [ ]:
df = sweep.to_pandas(metrics=("nextpnr.fmax", "nextpnr.lut")).rename(
    columns={
        "design.rtl.parameters.LOG2_DEPTH": "depth",
        "settings.nextpnr.seed": "seed",
        "settings.yosys_fpga.clock.period": "period",
        "nextpnr.fmax": "fmax",
        "nextpnr.lut": "lut",
    }
)
columns = ["depth", "seed", "period", "outcome", "error_code", "fmax", "lut"]
df[columns].to_csv(WORK / "fifo-sweep.csv", index=False)
df[columns]

To see why a point did not pass, read its result.

In [ ]:
for p in sweep:
    if not p.result.passed:
        print(p.index, p.result.outcome.value, p.result.error.code)
        print(p.result.error.message.splitlines()[0])

## Plot the Pareto front

A point is on the front when no other point has fewer LUTs and a higher `fmax`. The plot uses
Plotly and shows in the notebook. It does not export an image.

In [ ]:
passed = df[df.outcome == "passed"]
ranked = passed.sort_values(["lut", "fmax"], ascending=[True, False])
front = ranked[ranked.fmax > ranked.fmax.cummax().shift(fill_value=0)]
unit = sweep.points[0].result.nodes["nextpnr"].record.metric_info["fmax"].unit

fig = go.Figure()
fig.add_scatter(
    x=passed.lut,
    y=passed.fmax,
    mode="markers",
    name="passed",
    text=[f"depth {d}, seed {s}" for d, s in zip(passed.depth, passed.seed)],
    hovertemplate="%{text}<br>%{x} LUTs, %{y:.1f}<extra></extra>",
)
fig.add_scatter(
    x=front.lut, y=front.fmax, mode="lines", name="Pareto front", line={"shape": "hv"}
)
fig.update_layout(
    template="plotly_white",
    xaxis_title="LUTs",
    yaxis_title=f"fmax ({unit})",
    legend={"orientation": "h", "y": -0.2},
)
fig.show()

## Read the runs later

The sweep left its runs in the run root. `ws.records` reads them from the disk. It starts no tool
and takes no lock. The filters are `design_name`, `flow` and `target`. A filter that you leave out
matches everything. Failed runs are in the listing too.

In [ ]:
history = ws.records(design_name="fifo", flow="nextpnr")
print(len(history), "records,", len(history.problems), "problems")

pd.DataFrame(
    [
        {
            "run": record.run_dir.name,
            "outcome": record.outcome.value,
            "depth": record.configuration["rtl"]["parameters"].get("LOG2_DEPTH"),
            "seed": record.settings["seed"],
            "fmax": record.metrics.get("fmax"),
        }
        for record in history
    ]
).sort_values(["depth", "seed"])

`ws.load` reads the record of one run directory. `open` gives the files of the run. While the
`with` block runs, `open` holds a lease on the run directory, so no launch can replace the files.

In [ ]:
record = ws.load(sweep.points[0].result.nodes["nextpnr"].run_dir)
print(record.flow, record.outcome.value, record.metric_info["fmax"].unit)
with record.open() as run:
    print(sorted(run.artifacts))
    print(run.output("config").name)
    print(run.artifacts["log"].stat().st_size > 0)

A new run of the same configuration replaces the run directory. A record that you hold from
before does not describe the files any more, and `open` says so. Start the new run outside a
`with record.open()` block: a launch that starts while the thread holds a lease raises
`LeaseHeldError`.

In [ ]:
ws.run(specs[0], rebuild_all=True)
try:
    with record.open() as run:
        pass
except RunReplacedError as error:
    print(error.code)
    print(error.expected_run_id != error.found_run_id)

current = ws.load(record.run_dir)
print(current.run_id != record.run_id, current.identity == record.identity)

The old record is still valid data, but its files are gone. `ws.load` returns the record of the
new run. Both runs have the same configuration, so their `identity` is equal.

## Two cases with nothing to show

A run root with no runs gives an empty listing. It is no error.

In [ ]:
listing = xeda.Workspace(WORK / "empty").records(design_name="fifo")
if not listing.records:
    print("no runs yet: run a sweep first, or open a workspace on a run root that has runs")

A mistyped metric name is no error either. The column of that metric holds `None` for every
point. To find a mistake like this, compare the name with the metrics that the flow describes.

In [ ]:
name = "nextpnr.fmx"  # a typo for nextpnr.fmax
column = sweep.to_pandas(metrics=(name,))[name]
print("every value is empty:", column.isna().all())

node, _, metric = name.partition(".")
known = [entry["path"] for entry in results_info(node)["metrics"]]
if metric not in known:
    print(f"{metric!r} is not a metric of {node}. Its metrics are: {', '.join(known)}")